In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

from cpcv_analysis.config import XGB_PARAMS, ASSETS, TIMELINES, SYNTHETIC_SCENARIOS, TIMELINE_B
from cpcv_analysis.experiment import run_experiment, _run_experiment_from_arrays
from cpcv_analysis.synthetic import generate_synthetic_prices
from cpcv_analysis.data import build_features

clf_template = XGBClassifier(**XGB_PARAMS)
METHODS = ["cpcv", "wf", "kfold"]
ASSET_TICKERS = list(ASSETS.keys())

os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)
print(f"Assets: {ASSET_TICKERS}")
print(f"Timelines: {[t['name'] for t in TIMELINES]}")
print(f"Methods: {METHODS}")

Assets: ['SPY', 'AAPL', 'IWM', 'BTC-USD']
Timelines: ['2023-2026', 'COVID-2017-2020']
Methods: ['cpcv', 'wf', 'kfold']


In [2]:
real_rows = []

for timeline_cfg in TIMELINES:
    for ticker in ASSET_TICKERS:
        for method in METHODS:
            print(f"Running: {timeline_cfg['name']} | {ticker} | {method}...")
            try:
                metrics, fig = run_experiment(ticker, timeline_cfg, clf_template, method=method)
                fname = f"plots/real_{timeline_cfg['name']}_{ticker}_{method}.png"
                fig.savefig(fname, dpi=100, bbox_inches="tight")
                plt.close(fig)
                row = dict(timeline=timeline_cfg["name"], asset=ticker, method=method, **metrics)
                real_rows.append(row)
                print(f"  -> delta_median={metrics['delta_median']:.3f}, coverage_90={metrics['coverage_90']}")
            except Exception as e:
                print(f"  ERROR: {e}")
                import traceback; traceback.print_exc()

real_df = pd.DataFrame(real_rows)
real_df.to_csv("results/real_experiments.csv", index=False)
print(f"\nCompleted {len(real_rows)} real experiments")
real_df

Running: 2023-2026 | SPY | cpcv...


[data] Downloaded SPY prices and cached → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/SPY_2023-05-01_2026-01-01.csv
[data] Observations: 646  |  Class balance: 64.1% up
[data] Feature shape: (646, 6)  |  Date range: 2023-05-30 → 2025-12-23



Val SRs:        ['2.110', '1.521', '2.549', '2.762', '2.869']
Mediana val:    2.549
Hold-out SR:    0.945
[P5, P95]:      [1.639, 2.847]
Cobertura:      fuera IC
  -> delta_median=1.604, coverage_90=0
Running: 2023-2026 | SPY | wf...
[data] Loaded SPY prices from cache → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/SPY_2023-05-01_2026-01-01.csv
[data] Observations: 646  |  Class balance: 64.1% up
[data] Feature shape: (646, 6)  |  Date range: 2023-05-30 → 2025-12-23



Val SRs:        ['3.484', '2.583']
Mediana val:    3.034
Hold-out SR:    0.945
[P5, P95]:      [2.628, 3.439]
Cobertura:      fuera IC
  -> delta_median=2.088, coverage_90=0
Running: 2023-2026 | SPY | kfold...
[data] Loaded SPY prices from cache → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/SPY_2023-05-01_2026-01-01.csv
[data] Observations: 646  |  Class balance: 64.1% up
[data] Feature shape: (646, 6)  |  Date range: 2023-05-30 → 2025-12-23



Val SRs:        ['0.869', '13.282', '-1.025', '7.402', '2.731']
Mediana val:    2.731
Hold-out SR:    0.945
[P5, P95]:      [-0.646, 12.106]
Cobertura:      dentro IC
  -> delta_median=1.786, coverage_90=1
Running: 2023-2026 | AAPL | cpcv...


...

[data] Downloaded BTC-USD prices and cached → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/BTC-USD_2017-05-01_2020-05-01.csv
[data] Observations: 1071  |  Class balance: 53.4% up
[data] Feature shape: (1071, 6)  |  Date range: 2017-05-21 → 2020-04-25



Val SRs:        ['-4.233', '-4.048', '-2.983', '-2.095', '-5.329']
Mediana val:    -4.048
Hold-out SR:    -0.849
[P5, P95]:      [-5.110, -2.273]
Cobertura:      fuera IC
  -> delta_median=-3.199, coverage_90=0
Running: COVID-2017-2020 | BTC-USD | wf...
[data] Loaded BTC-USD prices from cache → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/BTC-USD_2017-05-01_2020-05-01.csv
[data] Observations: 1071  |  Class balance: 53.4% up
[data] Feature shape: (1071, 6)  |  Date range: 2017-05-21 → 2020-04-25



Val SRs:        ['-3.442', '0.735']
Mediana val:    -1.354
Hold-out SR:    -0.849
[P5, P95]:      [-3.233, 0.526]
Cobertura:      dentro IC
  -> delta_median=-0.504, coverage_90=1
Running: COVID-2017-2020 | BTC-USD | kfold...
[data] Loaded BTC-USD prices from cache → /Users/jeronimo.deli/Desktop/other/Vs/TSP/CPCV_tesis/cpcv_analysis/../data_cache/BTC-USD_2017-05-01_2020-05-01.csv
[data] Observations: 1071  |  Class balance: 53.4% up
[data] Feature shape: (1071, 6)  |  Date range: 2017-05-21 → 2020-04-25



Val SRs:        ['-2.396', '2.513', '-9.127', '-7.562', '0.198']
Mediana val:    -2.396
Hold-out SR:    -0.849
[P5, P95]:      [-8.814, 2.050]
Cobertura:      dentro IC
  -> delta_median=-1.546, coverage_90=1

Completed 24 real experiments


,timeline,asset,method,delta_median,coverage_90,rank_pct,dispersion,delta_maxDD,pct_positive
0,2023-2026,SPY,cpcv,1.604059,0,0.0,0.494423,-0.146693,1.0
1,2023-2026,SPY,wf,2.088433,0,0.0,0.450153,0.018513,1.0
2,2023-2026,SPY,kfold,1.785565,1,0.4,5.143395,-0.024489,0.8
3,2023-2026,AAPL,cpcv,-3.022427,0,1.0,0.797948,-1.116549,0.0
4,2023-2026,AAPL,wf,-1.164274,1,0.5,3.356836,-0.998752,0.5
5,2023-2026,AAPL,kfold,-2.213721,0,1.0,4.686517,-1.657106,0.0
6,2023-2026,IWM,cpcv,-0.111016,1,0.8,0.382312,-0.025103,1.0
7,2023-2026,IWM,wf,-0.148622,1,0.5,2.901554,-0.389705,0.5
8,2023-2026,IWM,kfold,-3.462283,1,0.6,2.504839,-0.447120,0.4
9,2023-2026,BTC-USD,cpcv,-2.508164,1,0.8,1.440239,-0.905531,0.4


In [3]:
display_cols = ["timeline", "asset", "method",
                "delta_median", "coverage_90", "rank_pct",
                "dispersion", "delta_maxDD", "pct_positive"]
real_df[display_cols].round(4)

,timeline,asset,method,delta_median,coverage_90,rank_pct,dispersion,delta_maxDD,pct_positive
0,2023-2026,SPY,cpcv,1.6041,0,0.0,0.4944,-0.1467,1.0
1,2023-2026,SPY,wf,2.0884,0,0.0,0.4502,0.0185,1.0
2,2023-2026,SPY,kfold,1.7856,1,0.4,5.1434,-0.0245,0.8
3,2023-2026,AAPL,cpcv,-3.0224,0,1.0,0.7979,-1.1165,0.0
4,2023-2026,AAPL,wf,-1.1643,1,0.5,3.3568,-0.9988,0.5
5,2023-2026,AAPL,kfold,-2.2137,0,1.0,4.6865,-1.6571,0.0
6,2023-2026,IWM,cpcv,-0.1110,1,0.8,0.3823,-0.0251,1.0
7,2023-2026,IWM,wf,-0.1486,1,0.5,2.9016,-0.3897,0.5
8,2023-2026,IWM,kfold,-3.4623,1,0.6,2.5048,-0.4471,0.4
9,2023-2026,BTC-USD,cpcv,-2.5082,1,0.8,1.4402,-0.9055,0.4


In [4]:
synth_rows = []

for scenario in SYNTHETIC_SCENARIOS:
    seed = int(scenario["id"])
    prices_synth = generate_synthetic_prices(scenario, seed=seed)
    X_s, y_s, t1_s, _, fwd_ret_s = build_features(prices_synth)

    timeline_synth = dict(
        name=f"synth_{scenario['id']}",
        wf_start=TIMELINE_B["wf_start"],
        dev_start=TIMELINE_B["dev_start"],
        dev_end=TIMELINE_B["dev_end"],
        retrain_start=TIMELINE_B["retrain_start"],
        retrain_end=TIMELINE_B["retrain_end"],
        holdout_start=TIMELINE_B["holdout_start"],
        holdout_end=TIMELINE_B["holdout_end"],
        download_start=TIMELINE_B["download_start"],
        download_end=TIMELINE_B["download_end"],
    )

    for method in METHODS:
        print(f"Synthetic {scenario['id']} {scenario['name']} | {method}...")
        try:
            metrics, fig = _run_experiment_from_arrays(
                X_s, y_s, t1_s, prices_synth, fwd_ret_s,
                timeline_synth, clf_template, method=method)
            fname = f"plots/synth_{scenario['id']}_{scenario['name']}_{method}.png"
            fig.savefig(fname, dpi=100, bbox_inches="tight")
            plt.close(fig)
            row = dict(scenario_id=scenario["id"], scenario_name=scenario["name"], method=method, **metrics)
            synth_rows.append(row)
            print(f"  -> delta_median={metrics['delta_median']:.3f}")
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()

synth_df = pd.DataFrame(synth_rows)
synth_df.to_csv("results/synthetic_experiments.csv", index=False)
print(f"\nCompleted {len(synth_rows)} synthetic experiments")

[data] Observations: 410  |  Class balance: 47.1% up
[data] Feature shape: (410, 6)  |  Date range: 2018-10-01 → 2020-04-24
Synthetic 01 bull_bull_bull | cpcv...



Val SRs:        ['3.449', '3.749', '4.440', '4.444', '2.204']
Mediana val:    3.749
Hold-out SR:    -5.637
[P5, P95]:      [2.453, 4.444]
Cobertura:      fuera IC
  -> delta_median=9.387
Synthetic 01 bull_bull_bull | wf...



Val SRs:        ['5.736', '-1.162']
Mediana val:    2.287
Hold-out SR:    -5.637
[P5, P95]:      [-0.817, 5.391]
Cobertura:      fuera IC
  -> delta_median=7.924
Synthetic 01 bull_bull_bull | kfold...



Val SRs:        ['13.885', '-0.814', '3.576', '1.025', '1.303']
Mediana val:    1.303
Hold-out SR:    -5.637
[P5, P95]:      [-0.446, 11.823]
Cobertura:      fuera IC
  -> delta_median=6.941
[data] Observations: 410  |  Class balance: 41.5% up
[data] Feature shape: (410, 6)  |  Date range: 2018-10-01 → 2020-04-24
Synthetic 02 bear_bear_bear | cpcv...



Val SRs:        ['2.646', '-0.309', '1.067', '-0.975', '0.800']
Mediana val:    0.800
Hold-out SR:    1.284
[P5, P95]:      [-0.841, 2.330]
Cobertura:      dentro IC
  -> delta_median=-0.484
Synthetic 02 bear_bear_bear | wf...


...


Val SRs:        ['-4.337', '-2.855']
Mediana val:    -3.596
Hold-out SR:    -3.327
[P5, P95]:      [-4.263, -2.930]
Cobertura:      dentro IC
  -> delta_median=-0.269
Synthetic 19 bear_crash_recovery | kfold...



Val SRs:        ['-5.298', '-5.832', '-4.049', '-3.916', '-5.008']
Mediana val:    -5.008
Hold-out SR:    -3.327
[P5, P95]:      [-5.725, -3.943]
Cobertura:      fuera IC
  -> delta_median=-1.681
[data] Observations: 410  |  Class balance: 42.0% up
[data] Feature shape: (410, 6)  |  Date range: 2018-10-01 → 2020-04-24
Synthetic 20 realistic_covid_analog | cpcv...



Val SRs:        ['4.660', '2.378', '2.191', '4.162', '4.751']
Mediana val:    4.162
Hold-out SR:    2.095
[P5, P95]:      [2.229, 4.733]
Cobertura:      fuera IC
  -> delta_median=2.068
Synthetic 20 realistic_covid_analog | wf...



Val SRs:        ['3.296', '-0.462']
Mediana val:    1.417
Hold-out SR:    2.095
[P5, P95]:      [-0.274, 3.108]
Cobertura:      dentro IC
  -> delta_median=-0.677
Synthetic 20 realistic_covid_analog | kfold...



Val SRs:        ['2.903', '3.639', '12.788', '-0.303', '3.055']
Mediana val:    3.055
Hold-out SR:    2.095
[P5, P95]:      [0.338, 10.958]
Cobertura:      dentro IC
  -> delta_median=0.961

Completed 60 synthetic experiments


In [5]:
display_cols2 = ["scenario_id", "scenario_name", "method",
                 "delta_median", "coverage_90", "rank_pct",
                 "dispersion", "delta_maxDD", "pct_positive"]
synth_df[display_cols2].round(4)

,scenario_id,scenario_name,method,delta_median,coverage_90,rank_pct,dispersion,delta_maxDD,pct_positive
0,01,bull_bull_bull,cpcv,9.3868,0,0.0,0.8241,0.2291,1.0
1,01,bull_bull_bull,wf,7.9243,0,0.0,3.4493,0.1809,0.5
2,01,bull_bull_bull,kfold,6.9407,0,0.0,5.2340,0.2131,0.8
3,02,bear_bear_bear,cpcv,-0.4843,1,0.8,1.2442,0.0004,0.6
4,02,bear_bear_bear,wf,-0.8670,0,1.0,0.0100,-0.0322,1.0
5,02,bear_bear_bear,kfold,-1.9604,1,0.6,4.5508,-0.3790,0.4
6,03,stagnant_all,cpcv,-2.6225,0,1.0,1.1418,-0.1043,0.4
7,03,stagnant_all,wf,-3.4472,0,1.0,1.8755,-0.2401,0.5
8,03,stagnant_all,kfold,-3.4137,1,0.8,4.6747,-0.2895,0.4
9,04,crash_in_dev_start,cpcv,8.1012,0,0.0,0.5126,0.2314,1.0


In [6]:
agg_cols = ["delta_median", "coverage_90", "rank_pct", "dispersion", "delta_maxDD", "pct_positive"]
summary_df = synth_df.groupby("method")[agg_cols].mean().round(4)
summary_df.index.name = "method"
summary_df.to_csv("results/summary_by_method.csv")

print("=== TABLE 3: Average calibration metrics per method (60 synthetic runs) ===")
display(summary_df)

=== TABLE 3: Average calibration metrics per method (60 synthetic runs) ===


,delta_median,coverage_90,rank_pct,dispersion,delta_maxDD,pct_positive
method,,,,,,
cpcv,1.6753,0.25,0.320,1.0498,-0.0662,0.610
kfold,1.5653,0.55,0.450,4.0480,-0.1789,0.580
wf,1.3223,0.35,0.425,1.6622,-0.0695,0.575


In [7]:
print(f"Real experiments: {len(real_df)} rows (expected 24)")
print(f"Synthetic experiments: {len(synth_df)} rows (expected 60)")
import glob
n_plots = len(glob.glob("plots/*.png"))
print(f"Figures saved: {n_plots} (expected 84)")
print(f"CSVs in results/: {os.listdir('results/')}")

assert len(real_df) == 24, f"Expected 24 real rows, got {len(real_df)}"
assert len(synth_df) == 60, f"Expected 60 synthetic rows, got {len(synth_df)}"
print("All assertions passed.")

Real experiments: 24 rows (expected 24)
Synthetic experiments: 60 rows (expected 60)
Figures saved: 89 (expected 84)
CSVs in results/: ['synthetic_experiments.csv', 'summary_by_method.csv', 'real_experiments.csv']
All assertions passed.
